# 03 - Feature Engineering

This notebook builds the complete feature set for bout prediction.

**CRITICAL:** All features use ONLY information available BEFORE each bout.

**Feature Categories:**
- Physical attributes (height, weight, BMI, age)
- Career statistics (win rate, bout count)
- Rating systems (ELO, Glicko-2)
- Current tournament state (wins, losses, streaks)
- Recent form (rolling win rates)
- Style profiles (push/grapple/evasion)
- Head-to-head history
- Pressure situations (kachikoshi, ozeki kadoban)
- Pairwise differentials
- Context features (venue, day)

**Inputs:**
- `matches_with_ratings.parquet` from notebook 02
- `rikishi.parquet` from notebook 01
- `rank_averages.parquet` from notebook 02

**Outputs:**
- `features.parquet` - Complete feature set for modeling

In [ ]:
# Environment setup
import sys
import os

INPUT_PATH = '/kaggle/input/sumo-data-02' if os.path.exists('/kaggle/input') else './output'
RIKISHI_PATH = '/kaggle/input/sumo-data-01' if os.path.exists('/kaggle/input') else './output'
OUTPUT_PATH = '/kaggle/working' if os.path.exists('/kaggle/working') else './output'

if not os.path.exists('/kaggle/input'):
    sys.path.insert(0, '../src')

print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")

In [ ]:
import pandas as pd
import numpy as np
import math
from collections import defaultdict, Counter
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
# Load matches with ratings
matches_df = pd.read_parquet(f"{INPUT_PATH}/matches_with_ratings.parquet")
print(f"Loaded {len(matches_df):,} matches with ratings")

# Load rikishi data
try:
    rikishi_df = pd.read_parquet(f"{RIKISHI_PATH}/rikishi.parquet")
    print(f"Loaded {len(rikishi_df):,} rikishi profiles")
except FileNotFoundError:
    print("Rikishi data not found, will use match data only")
    rikishi_df = None

# Load rank averages
try:
    rank_averages_df = pd.read_parquet(f"{INPUT_PATH}/rank_averages.parquet")
    print(f"Loaded rank averages for {len(rank_averages_df)} rank levels")
except FileNotFoundError:
    print("Rank averages not found")
    rank_averages_df = None

In [ ]:
print("\nMatch columns:")
print(matches_df.columns.tolist())
display(matches_df.head())

## Kimarite Categories

In [ ]:
KIMARITE_PUSH = [
    "oshidashi", "tsukidashi", "oshitaoshi", "tsukiotoshi",
    "tsukitaoshi", "okuridashi", "abisetaoshi"
]

KIMARITE_GRAPPLE = [
    "yorikiri", "uwatenage", "shitatenage", "sukuinage", "kotenage",
    "kubinage", "yoritaoshi", "uwatedashinage", "shitatedashinage",
    "kakenage", "kirikaeshi", "tsukaminage", "tsuridashi", "tsuriotoshi",
    "utchari", "sotogake", "uchigake", "kimedashi", "kimekiri",
    "katasukashi", "okurinage", "okuritaoshi", "okurihineri",
    "okuritsuridashi", "amiuchi", "sabaori", "waridashi", "makiotoshi",
    "uwatehineri", "shitatehineri"
]

KIMARITE_EVASION = [
    "hatakikomi", "hikiotoshi", "hikkake", "ketaguri", "kekaeshi",
    "ashitori", "tsumadori", "chongake", "kawazugake", "komatasukui",
    "tottari", "izori", "shumokuzori", "tasukizori", "nichonage"
]


def categorize_kimarite(kimarite: str) -> str:
    if not kimarite or pd.isna(kimarite):
        return "unknown"
    k = str(kimarite).lower().strip()
    if k in KIMARITE_PUSH:
        return "push"
    elif k in KIMARITE_GRAPPLE:
        return "grapple"
    elif k in KIMARITE_EVASION:
        return "evasion"
    return "grapple"

## Banzuke Rank Parser

In [ ]:
import re

def parse_banzuke_rank(rank_str: str) -> int:
    """Convert rank to numeric. Lower = higher rank."""
    if not rank_str or pd.isna(rank_str):
        return 999
    
    rank_str = str(rank_str).strip().upper()
    
    rank_bases = {
        "Y": 0, "O": 10, "S": 30, "K": 40, "M": 50, "J": 100
    }
    
    match = re.match(r"([YOSKM]|J)(\d+)?([EW])?", rank_str)
    if not match:
        return 999
    
    rank_letter = match.group(1)
    rank_num = int(match.group(2)) if match.group(2) else 1
    direction = match.group(3) if match.group(3) else "E"
    
    base = rank_bases.get(rank_letter, 999)
    numeric = base + (rank_num - 1) * 2
    if direction == "W":
        numeric += 1
    
    return numeric

# Test
print("Rank parsing examples:")
for rank in ['Y1E', 'Y1W', 'O1E', 'S1W', 'K1E', 'M1E', 'M10W', 'J1E']:
    print(f"  {rank} -> {parse_banzuke_rank(rank)}")

## Wrestler Stats Tracker

This class maintains rolling statistics for all wrestlers, processed chronologically.

In [ ]:
class WrestlerStatsTracker:
    """Tracks rolling stats for all wrestlers chronologically."""
    
    def __init__(self):
        # Career stats
        self.career_wins = defaultdict(int)
        self.career_losses = defaultdict(int)
        self.career_bouts = defaultdict(list)
        self.debut_date = {}
        
        # Tournament stats
        self.current_basho = ""
        self.basho_wins = defaultdict(int)
        self.basho_losses = defaultdict(int)
        
        # Streaks
        self.current_win_streak = defaultdict(int)
        self.current_loss_streak = defaultdict(int)
        
        # Completed basho records
        self.completed_basho_records = defaultdict(list)
        
        # Head to head
        self.h2h_wins = defaultdict(int)
        self.h2h_bouts = defaultdict(list)
        
        # Kimarite tracking
        self.win_kimarite = defaultdict(list)
        self.loss_kimarite = defaultdict(list)
    
    def new_basho(self, basho_id: str):
        """Called when a new tournament starts."""
        if self.current_basho:
            for wrestler_id in set(self.basho_wins.keys()) | set(self.basho_losses.keys()):
                wins = self.basho_wins.get(wrestler_id, 0)
                losses = self.basho_losses.get(wrestler_id, 0)
                if wins + losses > 0:
                    self.completed_basho_records[wrestler_id].append(
                        (self.current_basho, wins, losses)
                    )
        
        self.current_basho = basho_id
        self.basho_wins.clear()
        self.basho_losses.clear()
    
    def record_bout(self, bout: Dict):
        """Record a bout result."""
        east_id = bout['eastId']
        west_id = bout['westId']
        winner_id = bout.get('winnerId')
        basho_id = bout.get('bashoId', '')
        kimarite = bout.get('kimarite', '')
        
        if basho_id and basho_id != self.current_basho:
            self.new_basho(basho_id)
        
        if east_id not in self.debut_date:
            self.debut_date[east_id] = basho_id
        if west_id not in self.debut_date:
            self.debut_date[west_id] = basho_id
        
        # Record bout
        bout_record = {
            'basho': basho_id,
            'opponent': west_id,
            'won': winner_id == east_id if winner_id else None,
            'kimarite': kimarite
        }
        self.career_bouts[east_id].append(bout_record)
        self.career_bouts[west_id].append({
            **bout_record, 'opponent': east_id,
            'won': winner_id == west_id if winner_id else None
        })
        
        # H2H
        h2h_key = tuple(sorted([east_id, west_id]))
        self.h2h_bouts[h2h_key].append({
            'basho': basho_id, 'winner': winner_id, 'kimarite': kimarite
        })
        
        if winner_id:
            loser_id = west_id if winner_id == east_id else east_id
            
            self.career_wins[winner_id] += 1
            self.career_losses[loser_id] += 1
            self.basho_wins[winner_id] += 1
            self.basho_losses[loser_id] += 1
            
            self.current_win_streak[winner_id] += 1
            self.current_loss_streak[winner_id] = 0
            self.current_loss_streak[loser_id] += 1
            self.current_win_streak[loser_id] = 0
            
            if winner_id == east_id:
                self.h2h_wins[(east_id, west_id)] += 1
            else:
                self.h2h_wins[(west_id, east_id)] += 1
            
            if kimarite:
                self.win_kimarite[winner_id].append(kimarite.lower())
                self.loss_kimarite[loser_id].append(kimarite.lower())
    
    def get_career_stats(self, wrestler_id: int, bout_date: str = None) -> Dict:
        total_wins = self.career_wins.get(wrestler_id, 0)
        total_losses = self.career_losses.get(wrestler_id, 0)
        total_bouts = total_wins + total_losses
        
        career_length = 0
        debut = self.debut_date.get(wrestler_id)
        if debut and bout_date:
            try:
                debut_year = int(debut[:4])
                debut_month = int(debut[4:6])
                bout_year = int(bout_date[:4])
                bout_month = int(bout_date[4:6])
                career_length = (bout_year - debut_year) * 365 + (bout_month - debut_month) * 30
            except (ValueError, TypeError):
                pass
        
        return {
            'career_wins': total_wins,
            'career_losses': total_losses,
            'career_total_bouts': total_bouts,
            'career_win_rate': total_wins / total_bouts if total_bouts > 0 else 0.5,
            'career_length_days': max(0, career_length),
            'career_total_tournaments': len(self.completed_basho_records.get(wrestler_id, []))
        }
    
    def get_recent_form(self, wrestler_id: int, n_bouts: int = 10) -> Optional[float]:
        bouts = self.career_bouts.get(wrestler_id, [])
        if len(bouts) < n_bouts:
            return None
        recent = bouts[-n_bouts:]
        wins = sum(1 for b in recent if b.get('won') is True)
        return wins / n_bouts
    
    def get_basho_win_rates(self, wrestler_id: int, n_basho: int = 3) -> Optional[float]:
        records = self.completed_basho_records.get(wrestler_id, [])
        if len(records) < n_basho:
            return None
        recent = records[-n_basho:]
        total_wins = sum(r[1] for r in recent)
        total_bouts = sum(r[1] + r[2] for r in recent)
        return total_wins / total_bouts if total_bouts > 0 else None
    
    def get_kachikoshi_streak(self, wrestler_id: int) -> int:
        records = self.completed_basho_records.get(wrestler_id, [])
        streak = 0
        for _, wins, losses in reversed(records):
            if wins >= 8:
                streak += 1
            else:
                break
        return streak
    
    def get_makekoshi_streak(self, wrestler_id: int) -> int:
        records = self.completed_basho_records.get(wrestler_id, [])
        streak = 0
        for _, wins, losses in reversed(records):
            if wins < 8 and (wins + losses) >= 8:
                streak += 1
            else:
                break
        return streak
    
    def get_style_profile(self, wrestler_id: int) -> Dict:
        wins = self.win_kimarite.get(wrestler_id, [])
        losses = self.loss_kimarite.get(wrestler_id, [])
        
        def count_categories(kimarite_list):
            push = sum(1 for k in kimarite_list if k in KIMARITE_PUSH)
            grapple = sum(1 for k in kimarite_list if k in KIMARITE_GRAPPLE)
            evasion = sum(1 for k in kimarite_list if k in KIMARITE_EVASION)
            total = max(len(kimarite_list), 1)
            return push / total, grapple / total, evasion / total
        
        win_push, win_grapple, win_evasion = count_categories(wins)
        loss_push, loss_grapple, loss_evasion = count_categories(losses)
        
        if wins:
            win_counts = Counter(wins)
            modal = win_counts.most_common(3)
            modal_kimarite = modal[0][0] if modal else None
            second_modal = modal[1][0] if len(modal) > 1 else None
            third_modal = modal[2][0] if len(modal) > 2 else None
            pct_modal = modal[0][1] / len(wins) if modal else 0
            probs = [c / len(wins) for k, c in win_counts.items()]
            entropy = -sum(p * math.log(p) for p in probs if p > 0)
        else:
            modal_kimarite = second_modal = third_modal = None
            pct_modal = entropy = 0
        
        return {
            'pct_wins_by_push': win_push,
            'pct_wins_by_grapple': win_grapple,
            'pct_wins_by_evasion': win_evasion,
            'pct_losses_by_push': loss_push,
            'pct_losses_by_grapple': loss_grapple,
            'pct_losses_by_evasion': loss_evasion,
            'modal_kimarite': modal_kimarite,
            'second_modal_kimarite': second_modal,
            'third_modal_kimarite': third_modal,
            'pct_wins_by_modal_kimarite': pct_modal,
            'kimarite_entropy': entropy
        }
    
    def get_h2h_stats(self, wrestler_a: int, wrestler_b: int) -> Dict:
        h2h_key = tuple(sorted([wrestler_a, wrestler_b]))
        bouts = self.h2h_bouts.get(h2h_key, [])
        
        if not bouts:
            return {
                'h2h_total_bouts': 0, 'h2h_wins': 0, 'h2h_losses': 0,
                'h2h_win_rate': None, 'h2h_never_met': True,
                'h2h_current_streak': 0, 'h2h_last_result': None,
                'h2h_last_bout_kimarite': None
            }
        
        wins_a = sum(1 for b in bouts if b['winner'] == wrestler_a)
        losses_a = len(bouts) - wins_a
        
        streak = 0
        for b in reversed(bouts):
            if b['winner'] == wrestler_a:
                if streak >= 0:
                    streak += 1
                else:
                    break
            else:
                if streak <= 0:
                    streak -= 1
                else:
                    break
        
        last_bout = bouts[-1]
        last_result = 1 if last_bout['winner'] == wrestler_a else 0
        
        return {
            'h2h_total_bouts': len(bouts),
            'h2h_wins': wins_a,
            'h2h_losses': losses_a,
            'h2h_win_rate': wins_a / len(bouts),
            'h2h_never_met': False,
            'h2h_current_streak': streak,
            'h2h_last_result': last_result,
            'h2h_last_bout_kimarite': last_bout.get('kimarite')
        }
    
    def get_current_basho_stats(self, wrestler_id: int) -> Dict:
        return {
            'basho_wins': self.basho_wins.get(wrestler_id, 0),
            'basho_losses': self.basho_losses.get(wrestler_id, 0),
            'current_win_streak': self.current_win_streak.get(wrestler_id, 0),
            'current_loss_streak': self.current_loss_streak.get(wrestler_id, 0)
        }

## Feature Computation Functions

In [ ]:
def compute_pressure_features(row: Dict, basho_stats: Dict, prefix: str = 'east') -> Dict:
    """Compute pressure situation features."""
    wins = basho_stats.get('basho_wins', 0)
    losses = basho_stats.get('basho_losses', 0)
    day = row.get('day', 1)
    rank = row.get(f'{prefix}Rank', row.get(f'{prefix}_rank', ''))
    
    rank_str = str(rank).upper() if rank else ''
    is_ozeki = rank_str.startswith('O')
    is_yokozuna = rank_str.startswith('Y')
    
    return {
        f'{prefix}_needs_one_win_for_kachikoshi': int(wins == 7),
        f'{prefix}_needs_two_wins_for_kachikoshi': int(wins == 6),
        f'{prefix}_already_kachikoshi': int(wins >= 8),
        f'{prefix}_already_makekoshi': int(losses >= 8),
        f'{prefix}_is_day_15': int(day == 15),
        f'{prefix}_day_times_needs_one_for_kachikoshi': day if wins == 7 else 0,
        f'{prefix}_is_ozeki': int(is_ozeki),
        f'{prefix}_is_yokozuna': int(is_yokozuna),
        f'{prefix}_yokozuna_losing_record_so_far': int(is_yokozuna and losses > wins),
        f'{prefix}_yokozuna_multiple_losses_early': int(is_yokozuna and losses >= 2 and day <= 7),
    }


def compute_context_features(row: Dict) -> Dict:
    """Compute bout context features."""
    basho_id = str(row.get('bashoId', ''))
    
    try:
        year = int(basho_id[:4])
        month = int(basho_id[4:6])
    except (ValueError, IndexError):
        year, month = 0, 0
    
    month_to_num = {1: 1, 3: 2, 5: 3, 7: 4, 9: 5, 11: 6}
    month_to_venue = {1: 'Tokyo', 3: 'Osaka', 5: 'Tokyo', 7: 'Nagoya', 9: 'Tokyo', 11: 'Fukuoka'}
    
    venue = month_to_venue.get(month, 'Unknown')
    
    return {
        'year': year,
        'tournament_number': month_to_num.get(month, 0),
        'tournament_month': month,
        'venue': venue,
        'is_tokyo': int(venue == 'Tokyo'),
        'day_of_tournament': row.get('day', 0)
    }

## Main Feature Engineering

In [ ]:
def engineer_features(matches_df: pd.DataFrame,
                      rikishi_df: Optional[pd.DataFrame] = None,
                      rank_averages_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """Engineer all features, processing chronologically."""
    print("Starting feature engineering...")
    
    df = matches_df.copy()
    df = df.sort_values(['bashoId', 'day']).reset_index(drop=True)
    
    tracker = WrestlerStatsTracker()
    
    # Build lookups
    rikishi_lookup = {}
    if rikishi_df is not None:
        for _, row in rikishi_df.iterrows():
            rikishi_lookup[row.get('id')] = row.to_dict()
    
    rank_avg_lookup = {}
    if rank_averages_df is not None:
        for _, row in rank_averages_df.iterrows():
            rank_avg_lookup[row['rank_numeric']] = {
                'avg_elo': row.get('avg_elo_for_rank', 1500),
                'avg_glicko': row.get('avg_glicko_for_rank', 1500)
            }
    
    feature_rows = []
    n = len(df)
    
    for idx, row in df.iterrows():
        if idx % 25000 == 0:
            print(f"Processing bout {idx:,} / {n:,}")
        
        bout = row.to_dict()
        east_id = bout['eastId']
        west_id = bout['westId']
        basho_id = bout.get('bashoId', '')
        
        # Rikishi info
        east_info = rikishi_lookup.get(east_id, {})
        west_info = rikishi_lookup.get(west_id, {})
        
        # Initialize feature dict
        features = {
            'bout_id': bout.get('bout_id', idx),
            'bashoId': basho_id,
            'day': bout.get('day'),
            'eastId': east_id,
            'westId': west_id,
            'winnerId': bout.get('winnerId'),
            'kimarite': bout.get('kimarite'),
            'east_won': int(bout.get('winnerId') == east_id) if bout.get('winnerId') else None,
        }
        
        # Career stats
        for prefix, wrestler_id in [('east', east_id), ('west', west_id)]:
            career = tracker.get_career_stats(wrestler_id, basho_id)
            for k, v in career.items():
                features[f'{prefix}_{k}'] = v
        
        # Current basho stats
        east_basho = tracker.get_current_basho_stats(east_id)
        west_basho = tracker.get_current_basho_stats(west_id)
        
        for prefix, basho_stats in [('east', east_basho), ('west', west_basho)]:
            for k, v in basho_stats.items():
                features[f'{prefix}_{k}'] = v
        
        # Recent form
        for n_bouts in [5, 10, 15, 20]:
            features[f'east_win_rate_last_{n_bouts}_bouts'] = tracker.get_recent_form(east_id, n_bouts)
            features[f'west_win_rate_last_{n_bouts}_bouts'] = tracker.get_recent_form(west_id, n_bouts)
        
        # Basho win rates
        for n_basho in [1, 2, 3]:
            features[f'east_win_rate_last_{n_basho}_basho'] = tracker.get_basho_win_rates(east_id, n_basho)
            features[f'west_win_rate_last_{n_basho}_basho'] = tracker.get_basho_win_rates(west_id, n_basho)
        
        # Kachikoshi/Makekoshi streaks
        features['east_kachikoshi_streak'] = tracker.get_kachikoshi_streak(east_id)
        features['east_makekoshi_streak'] = tracker.get_makekoshi_streak(east_id)
        features['west_kachikoshi_streak'] = tracker.get_kachikoshi_streak(west_id)
        features['west_makekoshi_streak'] = tracker.get_makekoshi_streak(west_id)
        
        # Style profiles
        for prefix, wrestler_id in [('east', east_id), ('west', west_id)]:
            style = tracker.get_style_profile(wrestler_id)
            for k, v in style.items():
                features[f'{prefix}_{k}'] = v
        
        # H2H
        h2h = tracker.get_h2h_stats(east_id, west_id)
        for k, v in h2h.items():
            features[f'east_{k}'] = v
        
        # Pressure features
        features.update(compute_pressure_features(bout, east_basho, 'east'))
        features.update(compute_pressure_features(bout, west_basho, 'west'))
        
        # Rank features
        east_rank = bout.get('eastRank', bout.get('east_rank'))
        west_rank = bout.get('westRank', bout.get('west_rank'))
        features['east_rank_numeric'] = parse_banzuke_rank(east_rank)
        features['west_rank_numeric'] = parse_banzuke_rank(west_rank)
        features['rank_diff'] = features['west_rank_numeric'] - features['east_rank_numeric']
        
        # Rating vs expected for rank
        east_elo = bout.get('east_elo', 1500)
        west_elo = bout.get('west_elo', 1500)
        east_glicko = bout.get('east_glicko_rating', 1500)
        west_glicko = bout.get('west_glicko_rating', 1500)
        
        if features['east_rank_numeric'] in rank_avg_lookup:
            avg = rank_avg_lookup[features['east_rank_numeric']]
            features['east_elo_minus_expected'] = east_elo - avg['avg_elo']
            features['east_glicko_minus_expected'] = east_glicko - avg['avg_glicko']
        else:
            features['east_elo_minus_expected'] = 0
            features['east_glicko_minus_expected'] = 0
        
        if features['west_rank_numeric'] in rank_avg_lookup:
            avg = rank_avg_lookup[features['west_rank_numeric']]
            features['west_elo_minus_expected'] = west_elo - avg['avg_elo']
            features['west_glicko_minus_expected'] = west_glicko - avg['avg_glicko']
        else:
            features['west_elo_minus_expected'] = 0
            features['west_glicko_minus_expected'] = 0
        
        # Copy ratings
        features['east_elo'] = east_elo
        features['west_elo'] = west_elo
        features['east_glicko_rating'] = east_glicko
        features['west_glicko_rating'] = west_glicko
        features['east_glicko_rd'] = bout.get('east_glicko_rd')
        features['west_glicko_rd'] = bout.get('west_glicko_rd')
        features['elo_diff'] = east_elo - west_elo
        features['glicko_rating_diff'] = east_glicko - west_glicko
        
        # Pairwise diffs
        features['career_win_rate_diff'] = (
            features.get('east_career_win_rate', 0.5) - 
            features.get('west_career_win_rate', 0.5)
        )
        
        # Context
        features.update(compute_context_features(bout))
        
        feature_rows.append(features)
        
        # Record bout AFTER computing features
        tracker.record_bout(bout)
    
    print(f"Feature engineering complete. Generated {len(feature_rows):,} rows.")
    return pd.DataFrame(feature_rows)

In [ ]:
# Run feature engineering
features_df = engineer_features(matches_df, rikishi_df, rank_averages_df)

## Validate Features

In [ ]:
print(f"\nFeature DataFrame shape: {features_df.shape}")
print(f"\nColumns ({len(features_df.columns)}):")
print(features_df.columns.tolist())

In [ ]:
# Check for data leakage - features should have lower accuracy on first bouts
# (when we have no history)

# Split by career bout count
first_bouts = features_df[features_df['east_career_total_bouts'] < 10].copy()
later_bouts = features_df[features_df['east_career_total_bouts'] >= 50].copy()

print(f"First 10 career bouts: {len(first_bouts):,}")
print(f"After 50 career bouts: {len(later_bouts):,}")

# Check ELO predictive power
for name, df_subset in [('First bouts', first_bouts), ('Later bouts', later_bouts)]:
    valid = df_subset[
        df_subset['east_won'].notna() & 
        (df_subset['elo_diff'] != 0)
    ]
    if len(valid) > 0:
        accuracy = ((valid['elo_diff'] > 0) == (valid['east_won'] == 1)).mean()
        print(f"{name} - ELO accuracy: {accuracy:.1%}")

In [ ]:
# Feature summary
numeric_cols = features_df.select_dtypes(include=[np.number]).columns
print(f"\nNumeric features: {len(numeric_cols)}")
display(features_df[numeric_cols].describe().T)

In [ ]:
# Check missing values
missing = features_df.isnull().sum()
missing_pct = (missing / len(features_df) * 100).round(1)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['missing'] > 0].sort_values('pct', ascending=False)

print(f"\nColumns with missing values ({len(missing_df)}):")
display(missing_df.head(20))

## Save Features

In [ ]:
features_df.to_parquet(f"{OUTPUT_PATH}/features.parquet", index=False)
print(f"Saved {len(features_df):,} rows to {OUTPUT_PATH}/features.parquet")

In [ ]:
# Sample of final data
print("\nSample of engineered features:")
sample_cols = [
    'bashoId', 'day', 'east_won',
    'east_career_win_rate', 'west_career_win_rate', 'career_win_rate_diff',
    'east_elo', 'west_elo', 'elo_diff',
    'east_h2h_win_rate', 'east_h2h_total_bouts',
    'east_pct_wins_by_push', 'west_pct_wins_by_grapple'
]
display(features_df[sample_cols].tail(10))

## Next Steps

Features are ready for:
- `04_model_training.ipynb` - Train LightGBM models